# UCS420 Assignment 10 — Diabetes Care Information Assistant
**Name:** Aastha Tyagi  
**Roll No.:** 1024160131 (odd → Diabetes Care track)

Educational prototype only — not medical advice.


In [ ]:
# %pip install -q scikit-learn nltk pandas numpy matplotlib ipywidgets
import nltk, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from IPython.display import display, HTML, clear_output

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("setup ok")


## 1. Labelled corpus (≥ 40 queries, 7 intents)


In [ ]:
corpus_rows = [
    ("I keep feeling very thirsty and I urinate a lot. Could this be diabetes?", "symptoms_diagnosis"),
    ("What are the early warning signs of type 2 diabetes?", "symptoms_diagnosis"),
    ("Why do I have blurred vision and tingling in my feet?", "symptoms_diagnosis"),
    ("Difference between type 1 and type 2 diabetes?", "symptoms_diagnosis"),
    ("I have unexplained weight loss and constant fatigue. Should I get tested?", "symptoms_diagnosis"),
    ("What is prediabetes and how is it diagnosed?", "symptoms_diagnosis"),
    ("Does slow healing of cuts mean I might have diabetes?", "symptoms_diagnosis"),
    ("What does polydipsia and polyuria indicate in diabetes?", "symptoms_diagnosis"),
    ("What foods should a person with diabetes avoid?", "diet_nutrition"),
    ("Can I eat fruit if I have type 2 diabetes?", "diet_nutrition"),
    ("How do I count carbohydrates for my meals?", "diet_nutrition"),
    ("Is a low GI diet helpful for blood sugar control?", "diet_nutrition"),
    ("What is a healthy breakfast for someone with diabetes?", "diet_nutrition"),
    ("Are artificial sweeteners safer than sugar for diabetics?", "diet_nutrition"),
    ("How much rice or roti can I eat with diabetes?", "diet_nutrition"),
    ("Should I skip meals to keep my glucose low?", "diet_nutrition"),
    ("How does metformin help in type 2 diabetes?", "medication_insulin"),
    ("When is insulin started in type 2 diabetes?", "medication_insulin"),
    ("What is the difference between basal and bolus insulin?", "medication_insulin"),
    ("Can I stop my diabetes tablets if my sugar looks normal?", "medication_insulin"),
    ("Why do some people need insulin injections every day?", "medication_insulin"),
    ("What side effects should I know about for diabetes medicines?", "medication_insulin"),
    ("How should unused insulin pens be stored?", "medication_insulin"),
    ("Is it okay to change my insulin dose on my own?", "medication_insulin"),
    ("How often should I check my blood sugar at home?", "blood_glucose_monitoring"),
    ("What is a normal fasting blood glucose range?", "blood_glucose_monitoring"),
    ("What does HbA1c measure and what target is usual?", "blood_glucose_monitoring"),
    ("My glucometer reading is 180 after lunch. Is that high?", "blood_glucose_monitoring"),
    ("How do I use a continuous glucose monitor?", "blood_glucose_monitoring"),
    ("When should I test glucose if I feel unwell?", "blood_glucose_monitoring"),
    ("What is the difference between plasma glucose and capillary glucose?", "blood_glucose_monitoring"),
    ("Why do morning sugars sometimes look higher than night sugars?", "blood_glucose_monitoring"),
    ("Is walking enough exercise for type 2 diabetes?", "exercise_lifestyle"),
    ("Can I do gym workouts if I take insulin?", "exercise_lifestyle"),
    ("How does physical activity improve insulin sensitivity?", "exercise_lifestyle"),
    ("Should I eat a snack before exercising with diabetes?", "exercise_lifestyle"),
    ("Does smoking make diabetes complications worse?", "exercise_lifestyle"),
    ("How much sleep do people with diabetes need?", "exercise_lifestyle"),
    ("Can stress raise my blood sugar even if I eat well?", "exercise_lifestyle"),
    ("What lifestyle changes reverse prediabetes?", "exercise_lifestyle"),
    ("How does diabetes damage the kidneys over time?", "complications_prevention"),
    ("Why do people with diabetes need regular eye checkups?", "complications_prevention"),
    ("What is diabetic neuropathy and how can I prevent it?", "complications_prevention"),
    ("How can I take care of my feet if I have diabetes?", "complications_prevention"),
    ("Does diabetes increase the risk of heart disease?", "complications_prevention"),
    ("What vaccines are recommended for adults with diabetes?", "complications_prevention"),
    ("How often should I get my kidneys and cholesterol tested?", "complications_prevention"),
    ("Can good sugar control prevent retinopathy?", "complications_prevention"),
    ("My sugar is 48 and I am sweating and shaking. What should I do?", "emergency_care"),
    ("What are the symptoms of hypoglycemia?", "emergency_care"),
    ("Someone with diabetes fainted. How should I help?", "emergency_care"),
    ("What is diabetic ketoacidosis and when is it an emergency?", "emergency_care"),
    ("I have fruity breath, vomiting and very high sugar. Is this urgent?", "emergency_care"),
    ("How do I use the 15-15 rule for low blood sugar?", "emergency_care"),
    ("When should I call an ambulance for diabetes?", "emergency_care"),
    ("Can low blood sugar during the night be dangerous?", "emergency_care"),
]
df = pd.DataFrame(corpus_rows, columns=["query", "intent"])
print("queries:", len(df), "intents:", df["intent"].nunique())
print(df["intent"].value_counts().sort_index())
df.to_csv("1024160131_diabetes_corpus.csv", index=False)
df.head()


## 2–3. Taxonomy, concepts, relationships


In [ ]:
TAXONOMY = {
    "Clinical Understanding": {
        "Symptom recognition": ["thirst", "polyuria", "fatigue", "blurred vision"],
        "Diagnosis and types": ["type 1", "type 2", "prediabetes"],
    },
    "Daily Self-Management": {
        "Diet and carbohydrate counting": ["glycaemic index", "portions", "fruit"],
        "Physical activity and habits": ["walking", "sleep", "stress", "smoking"],
    },
    "Pharmacological Care": {
        "Oral antidiabetic medicines": ["metformin", "adherence"],
        "Insulin therapy": ["basal", "bolus", "storage"],
    },
    "Monitoring and Safety": {
        "Glucose tracking": ["glucometer", "CGM", "HbA1c"],
        "Acute emergencies": ["hypoglycemia", "DKA"],
    },
    "Long-term Outcomes": {
        "Microvascular complications": ["retinopathy", "nephropathy", "neuropathy"],
        "Macrovascular prevention": ["heart disease", "cholesterol"],
    },
}
INTENT_TO_TAXONOMY = {
    "symptoms_diagnosis": ("Clinical Understanding", "Symptom recognition / Diagnosis"),
    "diet_nutrition": ("Daily Self-Management", "Diet"),
    "exercise_lifestyle": ("Daily Self-Management", "Activity"),
    "medication_insulin": ("Pharmacological Care", "Tablets / Insulin"),
    "blood_glucose_monitoring": ("Monitoring and Safety", "Glucose tracking"),
    "emergency_care": ("Monitoring and Safety", "Emergencies"),
    "complications_prevention": ("Long-term Outcomes", "Complications"),
}
print("Major categories:", len(TAXONOMY))
for major, subs in TAXONOMY.items():
    print(major, "->", list(subs))

CONCEPTS = ["Diabetes Mellitus", "Pancreas", "Insulin", "Glucose", "HbA1c",
            "Hypoglycemia", "Hyperglycemia", "Metformin", "Carbohydrates",
            "Neuropathy", "Retinopathy", "Nephropathy", "Ketones", "Exercise"]
RELATIONS = [
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "lowers", "Glucose"),
    ("Carbohydrates", "raise", "Glucose"),
    ("Diabetes Mellitus", "associated_with", "Hyperglycemia"),
    ("Hypoglycemia", "is_acute_emergency_of", "Diabetes Mellitus"),
    ("Metformin", "treats", "Diabetes Mellitus"),
    ("Exercise", "improves_sensitivity_to", "Insulin"),
    ("HbA1c", "measures_average_of", "Glucose"),
    ("Hyperglycemia", "can_cause", "Neuropathy"),
    ("Hyperglycemia", "can_cause", "Retinopathy"),
    ("Hyperglycemia", "can_cause", "Nephropathy"),
    ("Ketones", "indicate_risk_of", "Diabetic Ketoacidosis"),
]
print(len(CONCEPTS), "concepts,", len(RELATIONS), "relations")
for s, r, o in RELATIONS:
    print(f"  {s} --{r}--> {o}")


## 4. NLP preprocessing


In [ ]:
STOPWORDS = set(stopwords.words("english")) - {"not", "no", "nor"}
LEMMATIZER = WordNetLemmatizer()

def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(LEMMATIZER.lemmatize(t) for t in tokens)

demo = "I am NOT feeling well and my sugars are very High!!"
print("Original :", demo)
print("Processed:", preprocess(demo))
df["processed"] = df["query"].apply(preprocess)
df.head()


## 5–6. TF-IDF + Logistic Regression, evaluation


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["processed"], df["intent"], test_size=0.30, random_state=13, stratify=df["intent"]
)
intent_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
intent_pipeline.fit(X_train, y_train)
y_pred = intent_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
print(f"Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}")
print(classification_report(y_test, y_pred, zero_division=0))
labels = list(intent_pipeline.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right"); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## 7–8. Knowledge base and rules


In [ ]:
KNOWLEDGE_BASE = {
    "symptoms_diagnosis": "Common symptoms: thirst, frequent urination, fatigue, blurred vision, slow healing, weight loss. Only a clinician can diagnose using fasting glucose, OGTT or HbA1c. Type 1 needs insulin; type 2 involves insulin resistance.",
    "diet_nutrition": "Build plates around vegetables, lean protein and measured carbohydrate. Fruit is fine in portions; skip sugary drinks. Do not skip meals if you use insulin or sulfonylureas.",
    "medication_insulin": "Metformin is a usual first tablet in type 2. Insulin is required in type 1 and sometimes added in type 2. Basal covers background needs, bolus covers meals. Never change a dose without the prescriber.",
    "blood_glucose_monitoring": "Many non-pregnant adults aim near 80–130 mg/dL fasting and under ~180 mg/dL after meals — personal targets differ. HbA1c reflects ~3 months. Check more often when ill or changing activity.",
    "exercise_lifestyle": "About 150 minutes/week of moderate activity plus 2–3 strength sessions improves insulin sensitivity. Carry rapid carbohydrate if you use insulin. Sleep, stress care and stopping smoking all help.",
    "complications_prevention": "Long-term high glucose can damage eyes, kidneys, nerves and vessels. Keep glucose, BP and cholesterol in target, inspect feet daily, and attend yearly eye and kidney tests.",
    "emergency_care": "Glucose under 70 mg/dL with sweating or confusion: if awake, 15 g fast carbohydrate and recheck in 15 minutes. If unconscious, do not force food; call emergency services. Vomiting + fruity breath + very high sugar may be DKA.",
}
EMERGENCY_PATTERNS = [
    r"\b(unconscious|fainted|faint|collapse|seizure)\b",
    r"\b(hypoglyc|low sugar|sugar (is )?(very )?low)\b",
    r"\b(ketoacidosis|dka|fruity breath|ketones)\b",
    r"\b(ambulance|chest pain)\b",
]
UNSUPPORTED_CUES = [r"\b(fracture|broken bone|tooth|bitcoin|python code)\b"]
DOSE_CHANGE_CUES = [r"\b(increase|decrease|double|stop|skip).{0,40}\b(insulin|metformin|dose|tablet)\b", r"\bon my own\b"]

def hits_any(text, patterns):
    t = text.lower()
    return any(re.search(p, t) for p in patterns)

def apply_rules(raw_query, intent, confidence):
    flags, extras = [], []
    if hits_any(raw_query, EMERGENCY_PATTERNS) or intent == "emergency_care":
        flags.append("EMERGENCY_ESCALATION")
        extras.append("EMERGENCY: if the person cannot swallow or is unconscious, call emergency services. If awake with suspected low sugar, give 15 g fast carbohydrate and recheck in 15 minutes.")
        intent = "emergency_care"
    if confidence < 0.38 and "EMERGENCY_ESCALATION" not in flags:
        flags.append("LOW_CONFIDENCE")
        extras.append("I am not fully sure; treat this as general information and confirm with a clinician.")
    if hits_any(raw_query, UNSUPPORTED_CUES) or (confidence < 0.22 and "EMERGENCY_ESCALATION" not in flags):
        flags.append("UNSUPPORTED_QUERY")
        extras.append("This looks outside diabetes-care scope (symptoms, diet, medicines, monitoring, lifestyle, complications, emergencies).")
    if hits_any(raw_query, DOSE_CHANGE_CUES):
        flags.append("MEDICATION_SAFETY")
        extras.append("Do not start, stop or change insulin/tablet doses on your own.")
    flags.append("MEDICAL_DISCLAIMER")
    extras.append("Educational information only, not personalised medical advice.")
    if re.search(r"\b(pregnan|gestational)\b", raw_query.lower()):
        flags.append("SPECIALIST_REFERRAL")
        extras.append("Pregnancy and diabetes need specialist care.")
    return intent, flags, extras
print("KB intents:", list(KNOWLEDGE_BASE))


## 9. Integrated assistant


In [ ]:
def classify_intent(raw_query):
    processed = preprocess(raw_query)
    proba = intent_pipeline.predict_proba([processed])[0]
    idx = int(np.argmax(proba))
    return processed, intent_pipeline.classes_[idx], float(proba[idx])

def answer_query(raw_query):
    processed, pred, conf = classify_intent(raw_query)
    final, flags, extras = apply_rules(raw_query, pred, conf)
    major, sub = INTENT_TO_TAXONOMY.get(final, ("Unknown", "Unknown"))
    if "UNSUPPORTED_QUERY" in flags and "EMERGENCY_ESCALATION" not in flags:
        kb = "I could not map this to a supported diabetes-care intent. Please rephrase."
    else:
        kb = KNOWLEDGE_BASE[final]
    related = [f"{s} {r} {o}" for s, r, o in RELATIONS if s.lower() in processed or o.lower() in processed][:4]
    text = (
        f"Predicted: {pred} ({conf:.2f})\nFinal: {final}\nTaxonomy: {major} → {sub}\n\n{kb}\n\n"
        + "Rules:\n" + "\n".join(f"- {e}" for e in extras)
        + "\n\nDomain facts:\n" + "\n".join(f"- {x}" for x in (related or RELATIONS[:3]))
    )
    return {"query": raw_query, "processed": processed, "predicted": pred, "final": final,
            "confidence": conf, "flags": flags, "response": text}

def print_answer(res):
    print("=" * 72)
    print("USER:", res["query"])
    print("NLP :", res["processed"])
    print("FLAGS:", ", ".join(res["flags"]))
    print(res["response"])
    print()
print("assistant ready")


## 10. Required tests + interactive UI


In [ ]:
tests = [
    ("symptoms_diagnosis", "I have been unusually thirsty and I keep waking up to pass urine."),
    ("diet_nutrition", "Which breakfast foods are better if I have type 2 diabetes?"),
    ("medication_insulin", "Can you explain how metformin works for blood sugar?"),
    ("blood_glucose_monitoring", "What HbA1c number do doctors usually aim for?"),
    ("exercise_lifestyle", "Will daily walking help my insulin work better?"),
    ("complications_prevention", "Why must people with diabetes get their eyes checked every year?"),
    ("emergency_care", "How do I treat hypoglycemia with the 15-15 rule?"),
    ("DIFFERENT WORDING", "I am always parched and running to the loo — is that a sugar problem?"),
    ("UNSUPPORTED", "I think I fractured my wrist, should I put a cast myself?"),
    ("LOW CONFIDENCE", "stuff happening not sure"),
    ("EMERGENCY", "He has diabetes, he collapsed and is unconscious right now."),
]
for label, q in tests:
    print("TEST:", label)
    print_answer(answer_query(q))


In [ ]:
import ipywidgets as widgets
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

box = widgets.Textarea(placeholder="Ask a diabetes-care question...", layout=widgets.Layout(width="100%", height="70px"))
ask = widgets.Button(description="Ask", button_style="primary")
clr = widgets.Button(description="Clear")
out = widgets.Output()
HISTORY = []

def on_ask(_):
    text = box.value.strip()
    if not text:
        return
    HISTORY.append(answer_query(text))
    box.value = ""
    with out:
        clear_output()
        html = ["<ol>"]
        for item in HISTORY:
            html.append(
                f"<li><b>Q:</b> {item['query']}<br><b>Intent:</b> {item['final']} "
                f"({item['confidence']:.2f})<br><pre style='white-space:pre-wrap'>{item['response']}</pre></li>"
            )
        html.append("</ol>")
        display(HTML("".join(html)))

def on_clr(_):
    HISTORY.clear()
    with out:
        clear_output(); print("Ask another question.")

ask.on_click(on_ask); clr.on_click(on_clr)
display(HTML("<h3>Diabetes Care Information Assistant</h3><p>Aastha Tyagi · 1024160131</p>"))
display(box, widgets.HBox([ask, clr]), out)
